# HiconaCooler

In [1]:
# Import packages
import os
import shutil
import tempfile

import polars as pl
import hicona as hi

# Data paths 
COOLER_PATH = "../data/4DNFIIG4IWKW_10k.cool"
PROMOTERS_BED = "../data/promoters.bed"
HMM_STATES = "../data/hmm_states.bed"

As seen in the workflow section, the main starting point for HiCONA are `.cool` files. `HiconaCooler` is the class which allows us to interface with these files. 

## Reading data

To create a `HiconaCooler` object, simply pass the path to a valid cooler. If working with `.mcool` files, remember to add the full path to your resolution of interest (e.i. `somefile.mcool::resolutions/10000`).

In [2]:
handle = hi.HiconaCooler(COOLER_PATH)

As the name suggests, `HiconaCooler` inherits from the `Cooler` class from the `cooler` package, therefore retaining all its methods and attributes. As such, `HiconaCooler` objects can always be used in place of `Cooler` objects, including when reading data. For instance:

In [3]:
handle.pixels().fetch("chr2")
# handle.bins() and handle.matrix() are also options.
# For details on these consult cooler's documentation.

,bin1_id,bin2_id,count
45323250,24897,24897,429
45323251,24897,24898,199
45323252,24897,24899,80
45323253,24897,24900,40
45323254,24897,24901,34
...,...,...,...
89192797,49114,271348,1
89192798,49114,273380,1
89192799,49114,287507,1
89192800,49114,302868,1


Since pixel tables can be quite big, especially when annotated, you are likely to read them in chunks or multiple times. When a full pass of the data is required for some use, the selectors from `Cooler` are a fast and convenient option. 

However, you might be interested only in a subset of non-contiguous pixels, or perhaps you want to perform some complex manipulation. In these cases, `HiconaCooler` provides two methods to simplify the handling of bins and pixels data, namely `get_pixel_table` and `get_bin_table`.

In [4]:
pix_table = handle.get_pixel_table("chr2")
bin_table = handle.get_bin_table()

These methods create instances of two classes `PixelTable` and `BinTable`, respectively. These classes are discussed in detail in the dedicated notebook, but in short, temporary copies of the data are saved on disk. These copies can still be iterated quickly and in chunks, but they allow for in place manipulation (filtering, annotation, normalization...) without affecting the original cooler file. This comes at the cost of an initial overhead for copying to disk, so it is not always the most optimal solution.

In [5]:
pix_table.get_dataframe("chr2:0-300000")

bin1_id,bin2_id,count
i64,i64,i32
24897,24897,429
24897,24898,199
24897,24899,80
24897,24900,40
24897,24901,34
…,…,…
24923,24924,199
24923,24925,107
24924,24924,458


## Bin annotation

`Cooler` allows to add bin annotations only if they are present in the bin table at file creation. The `HiconaCooler` class provides a function, `bin_annotation_add`, to add one or more bin annotations to a previously generated cooler file. Different combinations of parameters can be used depending on the type of annotation to add.

In [6]:
# Bed file containing promoter information
promoters = pl.read_csv(PROMOTERS_BED, separator="\t")
promoters.head()

chrom,start,end,gene
str,i64,i64,str
"""chr1""",0,19000,"""Some gene"""


In [7]:
# Work on a temporary copy of the file to avoid permanent changes
with tempfile.TemporaryDirectory() as tmp_dir_name:
    tmp_file = os.path.join(tmp_dir_name, os.path.basename(COOLER_PATH))
    shutil.copy2(COOLER_PATH, tmp_file)

    copy_handle = hi.HiconaCooler(tmp_file)
    copy_handle.bin_annot_add(promoters, metric="bp_overlap", consolidate=True)
    annot_bins = copy_handle.bins()[:]

annot_bins

,chrom,start,end,gene
0,chr1,0,10000,Some gene
1,chr1,10000,20000,Some gene
2,chr1,20000,30000,None
3,chr1,30000,40000,None
4,chr1,40000,50000,None
...,...,...,...,...
308832,chrY,57180000,57190000,None
308833,chrY,57190000,57200000,None
308834,chrY,57200000,57210000,None
308835,chrY,57210000,57220000,None


In [8]:
# ChromHMM like-annotation
hmm_states = pl.read_csv(HMM_STATES, separator="\t", new_columns=("chrom", "start", "end", "state"))
hmm_states.head()

chrom,start,end,state
str,i64,i64,str
"""chr1""",10800,13000,"""Void"""
"""chr1""",13000,13200,"""Enh"""
"""chr1""",13200,16000,"""Tx"""
"""chr1""",16000,17200,"""Tx"""
"""chr1""",17200,19200,"""Tx"""


In [9]:
# Work on a temporary copy of the file to avoid permanent changes
with tempfile.TemporaryDirectory() as tmp_dir_name:
    tmp_file = os.path.join(tmp_dir_name, os.path.basename(COOLER_PATH))
    shutil.copy2(COOLER_PATH, tmp_file)

    copy_handle = hi.HiconaCooler(tmp_file)
    copy_handle.bin_annot_add(hmm_states, metric="chrom_enrich", consolidate=True, save_all_mods=True)
    annot_bins = copy_handle.bins()[:]

annot_bins

,chrom,start,end,Enh_chrom_enrich,Het_chrom_enrich,Prom_chrom_enrich,Repr_Pc_chrom_enrich,Tx_chrom_enrich,Void_chrom_enrich,null_chrom_enrich
0,chr1,0,10000,NaN,NaN,NaN,NaN,NaN,NaN,3.362657
1,chr1,10000,20000,0.473073,NaN,NaN,NaN,1.984741,0.832186,0.801496
2,chr1,20000,30000,NaN,NaN,NaN,NaN,NaN,1.848506,NaN
3,chr1,30000,40000,NaN,NaN,NaN,NaN,NaN,1.848506,NaN
4,chr1,40000,50000,NaN,NaN,NaN,NaN,NaN,1.848506,NaN
...,...,...,...,...,...,...,...,...,...,...
308832,chrY,57180000,57190000,NaN,NaN,NaN,NaN,NaN,1.715164,NaN
308833,chrY,57190000,57200000,NaN,NaN,NaN,NaN,NaN,1.715164,NaN
308834,chrY,57200000,57210000,NaN,NaN,NaN,NaN,NaN,1.715164,NaN
308835,chrY,57210000,57220000,NaN,NaN,NaN,NaN,NaN,1.408816,0.582513


<div class=warn>
annotations can make the size of the cooler increase by a lot very quickly. In general, it is better to add the annotation to the cooler only when you are likely to use it multiple times. For one-time annotations, HiCONA allows to annotate `BinTable` objects directly.
</div>

In [10]:
from IPython.core.display import HTML

def css_styling():
    styles = open("../styles/custom.css", "r").read()
    return HTML(styles)
css_styling()